# SNAPTURE: six-material training on Colab Free
Optional GPU compute, not a recognition API. No API key or paid plan is required. Free GPU availability is not guaranteed: https://research.google.com/colaboratory/faq.html
Only upload a prepared training bundle when you have permission to send its images to Google. Remove faces, documents and identifying details first. Do not include accounts, scan histories, .env files or databases.
Prepare and package reviewed, physically grouped data with scripts/reviewed_dataset.py. Select Runtime > Change runtime type > GPU if available. Training creates a candidate only; independent metrics and content review are still required. It does not promote a model or promise 80% accuracy.

In [ ]:
import tensorflow as tf
print('TensorFlow:', tf.__version__)
print('Available GPUs:', tf.config.list_physical_devices('GPU'))
from google.colab import files
uploaded = files.upload()  # Select the reviewed training bundle, never the entire repository.
assert len(uploaded) == 1, 'Upload one training bundle.'

In [ ]:
from pathlib import Path
import zipfile, subprocess, sys
workspace = Path('/content/snapture_training')
assert not workspace.exists(), 'Restart with a fresh runtime to avoid mixing datasets.'
workspace.mkdir()
with zipfile.ZipFile(next(iter(uploaded))) as archive:
    for item in archive.infolist():
        target = (workspace / item.filename).resolve()
        assert target.is_relative_to(workspace), 'Unsafe archive path'
    archive.extractall(workspace)
subprocess.run([sys.executable, str(workspace/'scripts/reviewed_dataset.py'), 'validate', str(workspace/'dataset')], check=True)

In [ ]:
subprocess.run([sys.executable, str(workspace/'scripts/train_model.py'), '--data-dir', str(workspace/'dataset'), '--output-dir', str(workspace/'candidate'), '--epochs', '30'], check=True)

In [ ]:
import json, shutil
summary = json.loads((workspace/'candidate/training_summary.json').read_text())
print(json.dumps({'validation': summary['validation'], 'test': summary['test']}, indent=2))
bundle = shutil.make_archive('/content/snapture_candidate', 'zip', workspace/'candidate')
files.download(bundle)
# Keep the candidate separate locally. Never rename labels or auto-replace active_model.json.